# Med-DDPM v2 — Conditional Diffusion for Brain MRI Synthesis
Based on: Dorjsembe et al. 2024 — Conditional Diffusion Models for Semantic 3D Brain MRI Synthesis

Adapted from 3D → 2D for LGG MRI slices (256×256×3). Mask conditioning via channel-wise concatenation at every denoising step.

## Cell 1 — Drive mount and repo setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.chdir("/content/Mask-to-MRI")

# Point sys.path to project ROOT so `from src.med_ddpm_v2 import ...` works
import sys
sys.path.insert(0, "/content/Mask-to-MRI")
print("Working directory:", os.getcwd())

## Cell 2 — Install dependencies

In [ ]:
!pip install -q tifffile scikit-image pyyaml albumentations
print("Dependencies installed")

## Cell 3 — GPU check

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} ({gpu.total_memory/1e9:.1f} GB)")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("No GPU — switch Colab runtime to T4")

## Cell 4 — Sanity check (dataset + model)

In [ ]:
from src.med_ddpm_v2 import ConditionalDDPM, CONFIG
from src.dataset import build_dataloaders
import torch

# Dataset check
loaders = build_dataloaders(
    CONFIG["raw_dir"],
    batch_size=CONFIG["batch_size"],
    tumor_ratio=CONFIG["tumor_ratio"],
    seed=CONFIG["seed"],
)
mask, mri = next(iter(loaders["train"]))
print(f"Dataset OK — mask={mask.shape}, mri={mri.shape}")
print(f"MRI range: [{mri.min():.2f}, {mri.max():.2f}]")
print(f"Channel means: R={mri[:,0].mean():.3f}, G={mri[:,1].mean():.3f}, B={mri[:,2].mean():.3f}")

# Model check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ConditionalDDPM(CONFIG).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model OK — {n_params:.1f}M parameters")

# Forward pass check
mask_d = mask[:2].to(device)
mri_d = mri[:2].to(device)
loss = model(mri_d, mask_d)
print(f"Forward pass OK — loss={loss.item():.4f}")

# Sampling check (few steps only for speed)
fake = model.sample(mask_d[:1], ddim_steps=10)
print(f"Sampling OK — shape={fake.shape}, std={fake.std():.3f}")
print("All checks passed — ready to train")

## Cell 5 — Train from scratch

In [ ]:
from src.med_ddpm_v2.train import train
from src.med_ddpm_v2.config import CONFIG
from src.dataset import build_dataloaders
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaders = build_dataloaders(
    CONFIG["raw_dir"],
    batch_size=CONFIG["batch_size"],
    tumor_ratio=CONFIG["tumor_ratio"],
    seed=CONFIG["seed"],
)

model = ConditionalDDPM(CONFIG).to(device)

train(
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    model=model,
    config=CONFIG,
    device=device,
    resume_from=None,  # set to checkpoint path to resume
)

## Cell 6 — Resume Training

In [ ]:
# Run this cell instead of Cell 5 to resume from a checkpoint
from src.med_ddpm_v2.train import train
from src.med_ddpm_v2.config import CONFIG
from src.dataset import build_dataloaders
import torch
import glob
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaders = build_dataloaders(
    CONFIG["raw_dir"],
    batch_size=CONFIG["batch_size"],
    tumor_ratio=CONFIG["tumor_ratio"],
    seed=CONFIG["seed"],
)

model = ConditionalDDPM(CONFIG).to(device)

# Find latest checkpoint by epoch number (not lexicographic)
checkpoints = glob.glob(f"{CONFIG['checkpoint_dir']}/checkpoint_v2_epoch_*.pt")
latest = max(checkpoints, key=lambda p: int(Path(p).stem.split('_')[-1])) if checkpoints else None
print(f"Resuming from: {latest}")

train(
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    model=model,
    config=CONFIG,
    device=device,
    resume_from=latest,
)

## Cell 7 — Generate synthetic MRIs

In [ ]:
from src.med_ddpm_v2.sample import generate_synthetic
from src.med_ddpm_v2.config import CONFIG
import glob
from pathlib import Path

# Use latest checkpoint (sorted by epoch number)
checkpoints = glob.glob(f"{CONFIG['checkpoint_dir']}/checkpoint_v2_epoch_*.pt")
best = max(checkpoints, key=lambda p: int(Path(p).stem.split('_')[-1]))
print(f"Generating from: {best}")

generate_synthetic(
    checkpoint_path=best,
    output_dir=CONFIG["synthetic_dir"],
    config=CONFIG,
)

## Cell 8 — Save everything to Drive

In [ ]:
import shutil
from pathlib import Path

local = "/content/Mask-to-MRI/outputs_v2"
drive = "/content/drive/MyDrive/mask-to-mri/outputs_v2"

shutil.copytree(local, drive, dirs_exist_ok=True)
print(f"Saved outputs_v2 to Drive")

# Also save synthetic
local_syn = "/content/Mask-to-MRI/outputs_v2/synthetic"
drive_syn = "/content/drive/MyDrive/mask-to-mri/outputs_v2/synthetic"
if Path(local_syn).exists():
    shutil.copytree(local_syn, drive_syn, dirs_exist_ok=True)
    print(f"Saved synthetic to Drive")